# Symbolic regression of the learned potential (Sec. VI, Appendix D)

Reruns the PySR searches behind Sec. VI and Appendix D on the reference model `default_pure_AT_rd15`.
Cells are taken verbatim from the authors' notebook; only paths were adapted.

* **Runtime:** each search takes minutes; the seeded serial runs of Appendix D (last section) take tens of minutes.
* **Reproducibility:** the Appendix D operator controls on the learned potential are seeded (`random_state=0`, serial)
  and reproduce exactly. The other searches (positive control, main recovery, feature ablation, analytic operator
  controls) were run unseeded: a rerun gives an equivalent but not identical Pareto front. The fronts used in the paper
  are in `results/sr_outputs/`, and `scripts/evaluate_sr_fronts.py` recomputes every number of Sec. VI from them in
  seconds, without PySR. Reruns write to `results/sr_outputs_rerun/`.
* PySR is imported before torch (juliacall/torch import-order issue).

In [ ]:
import pysr   # must be imported before torch
import sys
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
LNN_DIR, WEIGHTS, DATA, RESULTS = REPO / "lnn", REPO / "weights", REPO / "data", REPO / "results"
sys.path.insert(0, str(LNN_DIR))

In [ ]:
# --- setup: load the pure-AT rd15 model + analytic ground truth --------------
# IMPORTANT: import pysr (-> juliacall) BEFORE torch, or the two can segfault.
import pysr  # noqa: F401  (initializes the Julia backend first)
import sys, os
sys.path.insert(0, str(LNN_DIR))
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from physical_systems import three_body, initialize_particle_system
from fixedkinetic_LNN import LNN
torch.serialization.add_safe_globals([LNN])
from file_exports import load_particle
from release_io import load_model_path as load_model

PROJ    = str(WEIGHTS)
RUN_DIR = f"{PROJ}/default_pure_AT_rd15"
OUT_DIR = str(RESULTS / "sr_outputs_rerun")   # the paper's fronts are in results/sr_outputs
os.makedirs(OUT_DIR, exist_ok=True)

NU    = 1.0     # AT coupling E0 of this run
R_MIN = 0.6     # training minimum-separation cutoff
R_MAX = 5.0     # training maximum pairwise distance 2.0
DEVICE, DTYPE = torch.device("cpu"), torch.float32

model = load_model(f"{RUN_DIR}/model.pt", DEVICE, DTYPE); model.eval()
particle = load_particle(initialize_particle_system("three_body"),
                         f"{RUN_DIR}/particle.json")
particle.E0 = NU
print("Loaded. E0(nu)=", particle.E0, " eps_LJ=", particle.eps_LJ)

In [ ]:
# --- core evaluators (feature extraction mirrors CombinedNetwork.forward) -----
def features_from_q6(q6):
    """The exact 6 features the MLP builds internally: (r12,r23,r31,c1,c2,c3)."""
    p = np.asarray(q6, float).reshape(3, 2)
    r12 = np.linalg.norm(p[0]-p[1]); r23 = np.linalg.norm(p[1]-p[2]); r31 = np.linalg.norm(p[2]-p[0])
    v12 = p[1]-p[0]; v13 = p[2]-p[0]; v23 = p[2]-p[1]
    c1 = (v12 @ v13) / (r12*r31 + 1e-6)
    c2 = (-v12 @ v23) / (r12*r23 + 1e-6)
    c3 = ((-v13) @ (-v23)) / (r31*r23 + 1e-6)
    return np.array([r12, r23, r31, c1, c2, c3])

def learned_V(q6):
    """Learned potential: kinetic term vanishes when only q (6-vector) is fed."""
    s = torch.as_tensor(q6, dtype=DTYPE).reshape(1, 6)
    with torch.no_grad():
        return -model.lagrangian(s).item()

def true_V(q6):
    q = torch.as_tensor(q6, dtype=DTYPE).reshape(6)
    return float(particle.potential(q, torch.zeros(6, dtype=DTYPE)))

FEATURE_NAMES = ["r12", "r23", "r31", "c1", "c2", "c3"]

In [ ]:
# --- sample valid training-domain triangles and build (X, y) ------------------
# Same acceptance rule as the ensemble sampler: uniform points, keep the triangle
# only if every side is within the trained band [R_MIN, R_MAX]. Centered on COM.
def sample_configs(n, seed=0, box=1.4):
    rng = np.random.default_rng(seed)
    Q = []
    while len(Q) < n:
        pts = rng.uniform(-box, box, size=(3, 2))
        r = [np.linalg.norm(pts[0]-pts[1]),
             np.linalg.norm(pts[1]-pts[2]),
             np.linalg.norm(pts[2]-pts[0])]
        if min(r) < R_MIN or max(r) > R_MAX:
            continue
        Q.append((pts - pts.mean(0)).reshape(-1).astype(np.float32))
    return np.array(Q)

N_SAMPLES = 3000
Q  = sample_configs(N_SAMPLES, seed=0)
# DataFrame so PySR prints variables as r12,r23,... instead of x0,x1,...
X  = pd.DataFrame([features_from_q6(q) for q in Q], columns=FEATURE_NAMES)
yL = np.array([learned_V(q) for q in Q])   # learned potential  (SR target)
yT = np.array([true_V(q)    for q in Q])   # analytic potential (control target)

gauge = np.median(yL - yT)  # additive constant the learned V carries (forces-only training)
print(f"N={len(X)}  gauge=median(V_learn - V_true)={gauge:.4f}")
print(f"V_learned range [{yL.min():.3f}, {yL.max():.3f}] | V_true range [{yT.min():.3f}, {yT.max():.3f}]")
print(f"corr(V_learn, V_true) = {np.corrcoef(yL, yT)[0,1]:.6f}")

# closed-form AT reconstructed from the features, as an internal check
Xv = X.to_numpy()
phi = 1 + 3*Xv[:,3]*Xv[:,4]*Xv[:,5]
V_AT_closed = NU * phi / (Xv[:,0]*Xv[:,1]*Xv[:,2])**3
print("max|V_true - closed-form AT(features)| =", np.abs(yT - V_AT_closed).max())
print("max|V_learned - closed-form AT(features)| =", np.abs(yL - V_AT_closed).max())

#so, a correlation near 1 means the two curves have essentially the same ups and downs, peaks and valleys, and ordering of points—they differ mainly by an overall scale and/or offset (y=ax+b, precisely, but y-x are still mathematically linear! )


## PySR configuration

Operators chosen to *span* the AT form without handing it over: `+ - * /` plus a
`cube` and `inv` unary (so `(r12*r23*r31)^3` in the denominator is reachable as
`cube(inv(r12*r23*r31))` or `inv(cube(...))`, and the angular numerator
`1 + 3*c1*c2*c3` is a couple of `*`/`+`). We deliberately do **not** expose the
side lengths only — the cosines are provided so the compact AT numerator is
findable. Same config is reused for the control and the learned fits.

In [ ]:
from pysr import PySRRegressor

# A RELATIVE loss is essential: V spans ~5 orders of magnitude over the domain
# (s^-9 core vs. tail), so a plain squared loss fits only the small-r core and
# returns a messy rational. The relative loss makes every scale count, and the AT
# law drops out. The +0.3 floor keeps the loss finite through V's zero-crossings.
def make_regressor(niterations=100, populations=40, maxsize=30, **kw):
    return PySRRegressor(
        niterations=niterations,
        populations=populations,
        ncycles_per_iteration=700,
        binary_operators=["+", "-", "*", "/"],
        unary_operators=["cube(x) = x^3", "inv(x) = 1/x"],
        extra_sympy_mappings={"cube": lambda x: x**3, "inv": lambda x: 1/x},
        maxsize=maxsize,
        parsimony=0.001,
        elementwise_loss="loss(p, y) = ((p - y) / (abs(y) + 0.3))^2",
        model_selection="best",
        temp_equation_file=True, progress=False, verbosity=0,
        **kw,
    )

---
## Recovering the *learned* potential well — real validation data, symmetry, non-uniqueness

The runs above hand-sample triangles and (with plain settings) return a messy, slightly
asymmetric fit for $V_\mathrm{learned}$. Three changes turn that into a clean, symmetric
recovery, and expose a subtlety worth stating in the paper:

1. **Use the real rd15 validation set** (`threebody_24k_maxr_5_minr06_d90_rd15_...`) as the
   input distribution — positions only, evaluated at $\dot q=0$. It is dense in the small-$r$
   core (≈79% of configs have a side $<1$) yet reaches $r=5$.
2. **Search harder.** That distribution's extreme small-$r$ skew makes the loss landscape
   harder: with 100 iterations even the *analytic* control fails (R²≈0.86), but at **250
   iterations** (or with min-side balancing) the control recovers **exact AT** again. So the
   earlier valset failure was under-powered search, not bad data.
3. **Enforce the particle-relabel ($S_3$) symmetry by augmentation.** For each config we add
   its 6 label permutations, *each with its own* $V_\mathrm{learned}$ (we do **not** assume the
   network is invariant). This pushes the search toward the symmetric form AT should have —
   without ever handing it the functional form.

In [ ]:
import itertools
DS_VAL = f"{DATA}/threebody_24k_maxr_5_minr06_d90_rd15_validation_dataset.pt"
Dq = torch.load(DS_VAL, weights_only=False)[0].numpy()[:, :6]   # positions only
_perm = np.random.default_rng(0).permutation(len(Dq))

def augment_S3(Q):
    """x6: apply every particle-relabeling to the 3 particle positions."""
    out = []
    for q in Q:
        p = q.reshape(3, 2)
        for pm in itertools.permutations(range(3)):
            out.append(p[list(pm)].reshape(-1).astype(np.float32))
    return np.array(out)

Qbase = Dq[_perm[:1000]]
Qaug  = augment_S3(Qbase)
Qsel  = Qaug[np.random.default_rng(1).permutation(len(Qaug))[:5000]]

Xv  = pd.DataFrame([features_from_q6(q) for q in Qsel], columns=FEATURE_NAMES)
yLv = np.array([learned_V(q) for q in Qsel])
# fresh, unaugmented held-out valset configs
Qtev = Dq[_perm[6000:7500]]
Xtev = pd.DataFrame([features_from_q6(q) for q in Qtev], columns=FEATURE_NAMES)
yLtev = np.array([learned_V(q) for q in Qtev])
yTtev = np.array([true_V(q)    for q in Qtev])
print(f"valset base=1000  augmented={len(Qaug)}  used={len(Qsel)}")

In [ ]:
# strong-search, symmetry-augmented free-form fit of the LEARNED potential
sr_recover = make_regressor(niterations=250, populations=60)
sr_recover.fit(Xv, yLv)

p = sr_recover.predict(Xtev)
def _r2(y, q): return 1 - np.sum((y-q)**2)/np.sum((y-y.mean())**2)
print("Recovered learned-V expression (symmetry-augmented, valset):")
print("  sympy:", sr_recover.sympy())
print(f"  R^2(fit vs learned) = {_r2(yLtev, p):.4f}   R^2(vs TRUE AT) = {_r2(yTtev, p):.4f}")
sr_recover.equations_.to_csv(f"{OUT_DIR}/eqs_learned_recovered.csv", index=False)
sr_recover

**Reading cell 18's output.** PySR prints the result *factored*, which is why it doesn't look like AT at a glance. Expanded, it is

$$V_\mathrm{learned}\;\approx\;0.899\;+\;\frac{2.993\,c_1c_2c_3\;+\;0.997}{(r_{12}r_{23}r_{31})^3}\;+\;0.0068\,(r_{23}+r_{31})$$

i.e. Axilrod–Teller $(1+3c_1c_2c_3)/(r_{12}r_{23}r_{31})^3$ with coefficients $\approx(1,3)$, the gauge constant $0.899$, and a tiny residual asymmetry. To read whichever representative a rerun finds, evaluate `import sympy as sp; sp.expand(sr_recover.sympy())`.


Note that I DO NOT USE these results, though pretty similar, bc the entire augmentation discussion here is omitted from paper, bc it's not useful really.

### On-shell non-uniqueness check

The recovered expression is only *one representative* of an equivalence class. Across runs it comes out sometimes as the canonical AT **product** $c_1c_2c_3$ and sometimes in the cosine **sum** $(c_1+c_2+c_3)$ — both are the *same function* on valid triangles. The reason is that the six features are redundant: for any triangle $c_1^2+c_2^2+c_3^2+2c_1c_2c_3=1$, so algebraically distinct forms coincide on the manifold of valid triangles. We verify the recovered form and the canonical AT product form agree on-shell (correlation $\approx 1$ regardless of which representative was found).

In [ ]:
# compare: recovered learned form  vs  canonical AT product form  vs  true / learned V
Fv = Xtev.to_numpy()
Pv = Fv[:,0]*Fv[:,1]*Fv[:,2]
AT_product = NU*(1 + 3*Fv[:,3]*Fv[:,4]*Fv[:,5]) / Pv**3   # canonical AT (what the control finds)
recovered  = sr_recover.predict(Xtev)                     # what SR found for the learned V

ident = np.mean(Fv[:,3]**2 + Fv[:,4]**2 + Fv[:,5]**2 + 2*Fv[:,3]*Fv[:,4]*Fv[:,5])
print(f"triangle identity  <c1^2+c2^2+c3^2+2 c1c2c3> = {ident:.4f}  (=1 exactly on valid triangles)")
print(f"recovered vs learned V        : R^2 = {_r2(yLtev, recovered):.4f}")
print(f"recovered vs true AT          : R^2 = {_r2(yTtev, recovered):.4f}")
print(f"recovered vs AT-product form  : corr = {np.corrcoef(recovered, AT_product)[0,1]:.5f}  (on-shell equivalence)")

fig, ax = plt.subplots(1, 2, figsize=(8, 3.8))
ax[0].scatter(yTtev, recovered, s=7, alpha=0.4, color="#0072B2")
lo, hi = min(yTtev.min(), recovered.min()), max(yTtev.max(), recovered.max())
ax[0].plot([lo,hi],[lo,hi],"k--",lw=1)
ax[0].set_xlabel("true AT  $V$"); ax[0].set_ylabel("recovered learned form")
ax[0].set_title("recovered $\\approx$ true AT", fontsize=9)
ax[1].scatter(AT_product, recovered, s=7, alpha=0.4, color="#D55E00")
lo, hi = min(AT_product.min(), recovered.min()), max(AT_product.max(), recovered.max())
ax[1].plot([lo,hi],[lo,hi],"k--",lw=1)
ax[1].set_xlabel("canonical AT product form"); ax[1].set_ylabel("recovered (cos-sum) form")
ax[1].set_title("same function on-shell", fontsize=9)
fig.tight_layout(); plt.show()

---
## Why the cosines are necessary — feature ablation at equal search

Are the angular features doing real work, or could the sides alone suffice (they *are*
information-complete, since the cosines are functions of the sides)? We fit the **analytic**
$V_\mathrm{true}$ — isolating *feature sufficiency* from network quality — on the well-sampled
core $r\in[0.6,2]$ (where the search converges cleanly), once with `r`-only and once with all
six features, at the same budget.

In [ ]:
# feature ablation on the analytic potential (well-conditioned core), equal search
def _sample_core(n, seed, rmin=0.6, rmax=2.0, box=1.4):
    rng=np.random.default_rng(seed); Q=[]
    while len(Q)<n:
        pts=rng.uniform(-box,box,size=(3,2))
        r=[np.linalg.norm(pts[0]-pts[1]),np.linalg.norm(pts[1]-pts[2]),np.linalg.norm(pts[2]-pts[0])]
        if min(r)<rmin or max(r)>rmax: continue
        Q.append((pts-pts.mean(0)).reshape(-1).astype(np.float32))
    return np.array(Q)
_Qc = _sample_core(3000, 0);  _Qct = _sample_core(1200, 7)
_Ftr = np.array([features_from_q6(q) for q in _Qc]);  _ytr = np.array([true_V(q) for q in _Qc])
_Fte = np.array([features_from_q6(q) for q in _Qct]); _yte = np.array([true_V(q) for q in _Qct])
def _r2(y, p): return 1 - np.sum((y-p)**2)/np.sum((y-y.mean())**2)
for _tag, _cols in [("r-only (r12,r23,r31)", [0,1,2]), ("full-6 (r + cos)", [0,1,2,3,4,5])]:
    _names=[FEATURE_NAMES[k] for k in _cols]
    _sr=make_regressor(niterations=150, populations=50)
    _sr.fit(pd.DataFrame(_Ftr[:,_cols],columns=_names), _ytr)
    _p=_sr.predict(pd.DataFrame(_Fte[:,_cols],columns=_names))
    print(f"{_tag:24s}  R^2 = {_r2(_yte,_p):.4f}   best: {str(_sr.sympy())[:80]}")
print("\n-> r-only plateaus (radial only, no angular term); full-6 recovers exact AT.")

## Handling the redundancy elegantly — reduction modulo the triangle-identity ideal

The recovered form is unique only up to the constraint $g\equiv
c_1^2+c_2^2+c_3^2+2c_1c_2c_3-1=0$. Rather than say "we cannot get *the* exact form," we make it
precise: expressions that differ by a multiple of $g$ are equal on every valid triangle, i.e.
equal in the quotient ring modulo the ideal $\langle g\rangle$. A Gröbner basis gives a
**canonical representative**, turning the numerical "corr $\approx1$" into an exact statement.

In [ ]:
import sympy as sp
c1, c2, c3 = sp.symbols("c1 c2 c3", real=True)
g  = c1**2 + c2**2 + c3**2 + 2*c1*c2*c3 - 1          # triangle identity (=0 on valid triangles)
G  = sp.groebner([g], c1, c2, c3, order="grevlex", domain="QQ")

AT_num  = 1 + 3*c1*c2*c3                              # canonical AT angular numerator
alt_num = sp.Rational(5,2) - sp.Rational(3,2)*(c1**2 + c2**2 + c3**2)   # a different-looking rep

print("canonical form of  (1 + 3 c1c2c3)      mod <g>:", sp.expand(G.reduce(AT_num)[1]))
print("canonical form of  (5 - 3 sum c_i^2)/2 mod <g>:", sp.expand(G.reduce(alt_num)[1]))
print("difference AT_num - alt_num  =", sp.simplify((AT_num - alt_num)/g), "* g   (exact multiple)")
print("ideal-membership remainder of (AT_num - alt_num):", sp.expand(G.reduce(sp.expand(AT_num - alt_num))[1]))

def reduce_mod_identity(expr):
    """Canonical representative of a polynomial in (c1,c2,c3) on the triangle manifold."""
    return sp.expand(G.reduce(sp.expand(expr))[1])
print("\n-> both representatives share ONE canonical form; they are the same function on-shell.")

**Takeaway.** With equal search, the angular features are *necessary* (r-only saturates at
$R^2\!\approx\!0.4$ vs $R^2=1$ for the full set). And the residual ambiguity in the recovered
symbolic form is exactly the triangle-identity ideal $\langle g\rangle$: reducing modulo it
yields a single canonical Axilrod–Teller representative, so "recovered up to $\langle g\rangle$"
is a precise, defensible claim rather than a caveat.

---
## Operator-set robustness — is `cube` handing over the answer?

A fair objection: AT's radial part is $(r_{12}r_{23}r_{31})^3$, so giving PySR a `cube` operator makes the AT form low-complexity. Does the recovery depend on that choice? We refit the **analytic control** ($V_\mathrm{true}$, well-conditioned core — isolating the operator set, not network quality) under three operator sets:

1. **`cube` removed** — only `square`+`inv`, so SR must build $r^3$ itself (as $r\cdot r^2$): recovers AT, **$R^2=1.0$**.
2. **broadened** — `square`+`cube`+`inv`+`exp`: clean AT, **$R^2=1.0$** ($(c_1c_2c_3+\tfrac13)\cdot 3/(r_{12}r_{23}r_{31})^3$).
3. **generic** — additionally `sin`, i.e. transcendental *distractors*: **$R^2\approx0.99$** — the larger space makes the stochastic search slightly harder, so this run landed a near-AT local optimum rather than exactly AT, but the AT structure still dominates.

The decisive point is item 1: with `cube` **removed entirely**, SR still recovers AT *exactly* ($R^2=1$) by constructing the cubic itself. So `cube` is a convenience that lowers AT's complexity, not a crutch that enables it — the recovery reflects the data, not the operator list. (Binary `+ - * /` is already functionally complete for rational functions, and AT is rational in $(r,\cos\theta)$; the unary operators only change the complexity ranking.)

In [ ]:
# operator-set robustness on the analytic control (well-conditioned core)
def _core(n, seed, box=1.4):
    rng=np.random.default_rng(seed); Q=[]
    while len(Q)<n:
        p=rng.uniform(-box,box,size=(3,2))
        r=[np.linalg.norm(p[0]-p[1]),np.linalg.norm(p[1]-p[2]),np.linalg.norm(p[2]-p[0])]
        if min(r)<0.6 or max(r)>2.0: continue
        Q.append((p-p.mean(0)).reshape(-1).astype(np.float32))
    return np.array(Q)
_Qo=_core(3000,0); _Qot=_core(1200,7)
_Xo =pd.DataFrame([features_from_q6(q) for q in _Qo ],columns=FEATURE_NAMES)
_Xot=pd.DataFrame([features_from_q6(q) for q in _Qot],columns=FEATURE_NAMES)
_yo =np.array([true_V(q) for q in _Qo ]); _yot=np.array([true_V(q) for q in _Qot])
def _r2(y,p): return 1-np.sum((y-p)**2)/np.sum((y-y.mean())**2)
_sq={"square":lambda x:x**2}; _cb={"cube":lambda x:x**3}; _iv={"inv":lambda x:1/x}
_opsets={
 "1) no cube (square+inv)":            (["square(x)=x^2","inv(x)=1/x"],                 {**_sq,**_iv}),
 "2) broad (square+cube+inv+exp)":     (["square(x)=x^2","cube(x)=x^3","inv(x)=1/x","exp"], {**_sq,**_cb,**_iv}),
 "3) generic (+exp,sin distractors)":  (["square(x)=x^2","cube(x)=x^3","inv(x)=1/x","exp","sin"], {**_sq,**_cb,**_iv}),
}
for _tag,(_un,_map) in _opsets.items():
    _sr=PySRRegressor(niterations=200, populations=60, ncycles_per_iteration=700,
        binary_operators=["+","-","*","/"], unary_operators=_un, extra_sympy_mappings=_map,
        maxsize=35, parsimony=0.001, elementwise_loss="loss(p,y)=((p-y)/(abs(y)+0.3))^2",
        model_selection="best", temp_equation_file=True, progress=False, verbosity=0)
    _sr.fit(_Xo,_yo); _p=_sr.predict(_Xot)
    print(f"{_tag:36s} R^2(vs true AT)={_r2(_yot,_p):.4f}   best: {str(_sr.sympy())[:66]}")
print("\n-> AT recovered under every operator set: cube is a convenience, not a crutch.")

### Strongest case: **no unary operators at all**

With *only* `+ - * /` (zero unary operators — nothing that hints at powers), SR must spell the cubic out as repeated multiplication. It still recovers **exact AT** ($R^2=1$): the denominator $r_{12}^3r_{23}^3r_{31}^3$ is built as `r*r*r*...`. This is the clearest evidence that the operator set does not tailor the answer.

In [ ]:
# strongest case: NO unary operators — only + - * /
def _core2(n, seed, box=1.4):
    rng=np.random.default_rng(seed); Q=[]
    while len(Q)<n:
        p=rng.uniform(-box,box,size=(3,2))
        r=[np.linalg.norm(p[0]-p[1]),np.linalg.norm(p[1]-p[2]),np.linalg.norm(p[2]-p[0])]
        if min(r)<0.6 or max(r)>2.0: continue
        Q.append((p-p.mean(0)).reshape(-1).astype(np.float32))
    return np.array(Q)
_Qb=_core2(3000,0); _Qbt=_core2(1200,7)
_Xb =pd.DataFrame([features_from_q6(q) for q in _Qb ],columns=FEATURE_NAMES)
_Xbt=pd.DataFrame([features_from_q6(q) for q in _Qbt],columns=FEATURE_NAMES)
_yb =np.array([true_V(q) for q in _Qb ]); _ybt=np.array([true_V(q) for q in _Qbt])
_srb=PySRRegressor(niterations=300, populations=60, ncycles_per_iteration=700,
    binary_operators=["+","-","*","/"], unary_operators=[],
    maxsize=40, parsimony=0.001, elementwise_loss="loss(p,y)=((p-y)/(abs(y)+0.3))^2",
    model_selection="best", temp_equation_file=True, progress=False, verbosity=0)
_srb.fit(_Xb,_yb); _pb=_srb.predict(_Xbt)
print("BINARY-ONLY (+ - * /), no unary operators")
print(f"  R^2(vs true AT) = {1-np.sum((_ybt-_pb)**2)/np.sum((_ybt-_ybt.mean())**2):.4f}")
print("  best:", _srb.sympy())
print("  (the denominator is r12^3 r23^3 r31^3 spelled out as repeated multiplication)")

### Operator controls on the learned potential (seeded, full budget)

The trimmed serial rerun above (`populations=30, niterations=150`) was under-powered — $R^2\approx0.3$, far below the original unseeded run's $\approx0.95$ — the reduced population count starved the search, not the serial-vs-parallel switch itself. This cell keeps `deterministic=True, parallelism="serial"` (for reproducibility) but restores the original full budget (`populations=60, niterations=300`); slower, but a real search. **Not yet run** — serial mode at this budget takes a while; run when there's time/capacity for it.

In [ ]:
# operator controls on the LEARNED potential -- seeded, FULL budget, serial (slow but reproducible)
def _corel3(n, seed, box=1.4):
    rng=np.random.default_rng(seed); Q=[]
    while len(Q)<n:
        p=rng.uniform(-box,box,size=(3,2))
        r=[np.linalg.norm(p[0]-p[1]),np.linalg.norm(p[1]-p[2]),np.linalg.norm(p[2]-p[0])]
        if min(r)<0.6 or max(r)>2.0: continue
        Q.append((p-p.mean(0)).reshape(-1).astype(np.float32))
    return np.array(Q)
_Ql3=_corel3(3000,0); _Qlt3=_corel3(1200,7)
_Xl3 =pd.DataFrame([features_from_q6(q) for q in _Ql3 ],columns=FEATURE_NAMES)
_Xlt3=pd.DataFrame([features_from_q6(q) for q in _Qlt3],columns=FEATURE_NAMES)
_yLl3=np.array([learned_V(q) for q in _Ql3]); _yLlt3=np.array([learned_V(q) for q in _Qlt3])
_yTlt3=np.array([true_V(q) for q in _Qlt3])
def _r23(y,p): return 1-np.sum((y-p)**2)/np.sum((y-y.mean())**2)
_lvar3={
 "binary-only (no unary)": ([], {}),
 "no cube (square+inv)":    (["square(x)=x^2","inv(x)=1/x"], {"square":lambda x:x**2,"inv":lambda x:1/x}),
}
for _tag,(_un,_map) in _lvar3.items():
    _sr=PySRRegressor(niterations=300, populations=60, ncycles_per_iteration=700,
        binary_operators=["+","-","*","/"], unary_operators=_un, extra_sympy_mappings=_map,
        maxsize=40, parsimony=0.001, elementwise_loss="loss(p,y)=((p-y)/(abs(y)+0.3))^2",
        model_selection="best", deterministic=True, random_state=0, parallelism="serial",
        temp_equation_file=True, progress=False, verbosity=0)
    _sr.fit(_Xl3,_yLl3); _p=_sr.predict(_Xlt3)
    print("LEARNED V, %-24s R^2(fit vs learned)=%.4f  R^2(vs TRUE AT)=%.4f" % (_tag, _r23(_yLlt3,_p), _r23(_yTlt3,_p)))
    print("   full expr:", _sr.sympy())
print("-> seeded (random_state=0, serial): rerunning with these exact settings reproduces this exactly.")

In [ ]:
_sr.equations_.to_csv(f"{OUT_DIR}/eqs_learned_ops_nocube_seeded.csv", index=False)


In [ ]:
_sr_binary = PySRRegressor(
    niterations=300, populations=60, ncycles_per_iteration=700,
    binary_operators=["+","-","*","/"], unary_operators=[], extra_sympy_mappings={},
    maxsize=40, parsimony=0.001, elementwise_loss="loss(p,y)=((p-y)/(abs(y)+0.3))^2",
    model_selection="best", deterministic=True, random_state=0, parallelism="serial",
    temp_equation_file=True, progress=False, verbosity=0)
_sr_binary.fit(_Xl3, _yLl3)
_p = _sr_binary.predict(_Xlt3)
print("binary-only R^2(fit vs learned)=%.4f  R^2(vs TRUE AT)=%.4f" % (_r23(_yLlt3,_p), _r23(_yTlt3,_p)))
print("full expr:", _sr_binary.sympy())
_sr_binary.equations_.to_csv(f"{OUT_DIR}/eqs_learned_ops_binaryonly_seeded.csv", index=False)

# so, this is correct eqn + gauge + asymmetric(slightly larger than non-only-binary version, but still quite small), 

